# Session JSONLを1行ずつ見る
下のセルを実行し、パス欄にJSONLファイルのパスを入力して **読み込む** を押してください。
**前へ／次へ** で1行ずつ移動します。文章は改行をそのまま表示し、その下に元のJSONも表示します。ファイルの変更や外部への送信は行いません。

`ipywidgets` がない場合だけ、次のインストールセルの先頭の `#` を外して実行してください。

In [ ]:
# %pip install ipywidgets


In [ ]:
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

path = widgets.Text(
    value='/Users/cls-lab/Git/NEDO_RSI/WorldModel_plants/research/runs/20260915T045740Z_d0631e/experiments/A_01/agent/raw_sessions/rollout-2026-09-15T14-00-54-01a0a370-8009-73b2-977e-e9e44ea5ed68.jsonl',
    description="JSONLパス:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "initial"},
)
load = widgets.Button(description="読み込む", button_style="primary")
previous = widgets.Button(description="← 前へ", disabled=True)
next_line = widgets.Button(description="次へ →", disabled=True)
position = widgets.Label(value="未読み込み")
output = widgets.Output()
lines = []
index = 0


def show():
    previous.disabled = not lines or index == 0
    next_line.disabled = not lines or index == len(lines) - 1
    position.value = f"{index + 1} / {len(lines)} 行" if lines else "0 行"
    with output:
        clear_output(wait=True)
        if not lines:
            print("表示する行がありません。")
            return
        # JSONの構造を保ち、日本語と改行を読みやすく表示。
        # 文字列中の文章は別途そのまま表示する。
        try:
            record = json.loads(lines[index])
        except json.JSONDecodeError:
            print(lines[index])
            return

        def texts(value, location=""):
            if isinstance(value, dict):
                for key, child in value.items():
                    yield from texts(child, f"{location}.{key}" if location else key)
            elif isinstance(value, list):
                for n, child in enumerate(value):
                    yield from texts(child, f"{location}[{n}]")
            elif isinstance(value, str):
                yield location, value

        if isinstance(record, dict):
            print(f"type: {record.get('type', '')}    timestamp: {record.get('timestamp', '')}\n")
        for location, value in texts(record):
            print(f"[{location}]\n{value}\n")
        print("── 元のJSON（整形表示） ──")
        print(json.dumps(record, ensure_ascii=False, indent=2))


def read_file(_):
    global lines, index
    lines = []
    index = 0
    try:
        lines = Path(path.value.strip()).expanduser().read_text(encoding="utf-8").splitlines()
    except (OSError, UnicodeError) as error:
        previous.disabled = next_line.disabled = True
        position.value = "読み込み失敗"
        with output:
            clear_output(wait=True)
            print(str(error))
        return
    show()


def move(delta):
    global index
    if lines:
        index = max(0, min(len(lines) - 1, index + delta))
        show()


load.on_click(read_file)
previous.on_click(lambda _: move(-1))
next_line.on_click(lambda _: move(1))
display(path, widgets.HBox([load, previous, next_line, position]), output)
